In [1]:
import pandas as pd
import numpy as np
import re

# ==========================================
# 1. CONFIGURACIÓN INICIAL
# ==========================================
ARCHIVO_SAP = 'SAP Exports/DEC 2025.xlsx' 
CUENTA_OBJETIVO = '11136580'
MES_NOMBRE = "December" 
# Ajustado a doble cero antes del número [cite: 4]
MES_NUMERO = "012" 
ANIO = "2025"

# Orden de salida de las columnas solicitado
COLUMNAS_SALIDA = [
    'G/L Account', 'Journal Entry', 'Company Code', 'Journal Entry Type', 'Posting Date', 
    'Amount in Company Code Currency', 'Journal Entry Item Text', 
    'Value date', 'Amount in Transaction Currency', 
    'Offsetting Account', 'Amount in Global Currency', 'Assignment Reference', 
    'Clearing Date', 'Journal Entry Item', 'Fiscal Period', 'Clearing Journal Entry',
    'G/L Account Long Name', 'G/L Account Name'
]

reglas_entidades = {
    'Hoja 1': { 
        'prefix': 'Carlton BK', 
        'xq_ba': [r'BA.?0701', r'BA.?0702', r'BA.?1037', r'BA.?1038'], 
        'bank_text': [r'BE CASH DEPOSIT BAG NO', r'S\d+'] 
    }
}

# ==========================================
# 2. PROCESAMIENTO
# ==========================================

df_original = pd.read_excel(ARCHIVO_SAP)

def clean_amount(val):
    if pd.isna(val): return 0.0
    if isinstance(val, (int, float)): return float(val)
    val = str(val).replace('JMD', '').replace('USD', '').replace(',', '').strip()
    try: return float(val)
    except: return 0.0

# Formato de fecha Día/Mes/Año con barras
def clean_date(val):
    if pd.isna(val) or val == "" or str(val).lower() == "nan": return ""
    try:
        return pd.to_datetime(val).strftime('%d/%m/%Y')
    except:
        return str(val)

# Limpieza de montos
cols_monto = ['Amount in Transaction Currency', 'Amount in Global Currency', 'Amount in Company Code Currency']
for col in cols_monto:
    if col in df_original.columns:
        df_original[col] = df_original[col].apply(clean_amount)

df_acc = df_original[df_original['G/L Account'].astype(str).str.contains(CUENTA_OBJETIVO, na=False)].copy()

output_file = f'Master_{CUENTA_OBJETIVO}_{MES_NOMBRE}_{ANIO}.xlsx'
writer = pd.ExcelWriter(output_file, engine='xlsxwriter')

workbook = writer.book
fmt_jam = workbook.add_format({'num_format': '#,##0.00 "JMD"'}) 
fmt_usd = workbook.add_format({'num_format': '#,##0.00 "USD"'}) 
fmt_std = workbook.add_format({'num_format': '#,##0.00'})

for entidad, reglas in reglas_entidades.items():
    cond_xq_only = (df_acc['Journal Entry Type'].str.contains('XQ', na=False, case=False)) & \
                   (df_acc['Journal Entry Item Text'].str.contains('|'.join(reglas['xq_ba']), na=False, regex=True, case=False))
    
    cond_dz_only = (df_acc['Journal Entry Type'].str.contains('DZ', na=False, case=False))
    
    # Unimos ambas condiciones con un OR (|)
    condicion_xq = cond_xq_only | cond_dz_only
    df_xq = df_acc[condicion_xq].copy()
    
    condicion_bank = (df_acc['Journal Entry Type'].str.contains('BR|ZR|IB', na=False, case=False)) & \
                     (df_acc['Journal Entry Item Text'].str.contains('|'.join(reglas['bank_text']), na=False, regex=True, case=False))
    df_bank = df_acc[condicion_bank].copy()
    
    total_xq = df_xq['Amount in Transaction Currency'].sum()
    total_bank = df_bank['Amount in Transaction Currency'].sum()
    diferencia = total_xq + total_bank
    nom_text = f"{reglas['prefix']} Lib{abs(total_xq):.2f} BK{abs(total_bank):.2f} - {MES_NOMBRE} {ANIO}"
    
    filas_finales = df_xq.to_dict('records')
    filas_finales.append({}) 
    filas_finales.append({'Amount in Transaction Currency': total_xq})
    filas_finales.append({}) 
    
    filas_finales.extend(df_bank.to_dict('records'))
    filas_finales.append({}) 
    filas_finales.append({'Amount in Transaction Currency': total_bank})
    filas_finales.append({}) 
    filas_finales.append({
        'Amount in Transaction Currency': diferencia, 
        'Offsetting Account': nom_text 
    })
    
    df_sheet = pd.DataFrame(filas_finales)

    # Lógica para limpiar Fiscal Period y Company Code Currency en totales/vacíos
    def is_real_data(row):
        gl_val = str(row.get('G/L Account', ''))
        return pd.notna(row.get('G/L Account')) and gl_val != "" and gl_val.lower() != "nan"

    if 'Amount in Transaction Currency' in df_sheet.columns:
        # Solo poner JMD y 002 si hay una cuenta contable real en esa fila
        df_sheet['Amount in Company Code Currency'] = df_sheet.apply(
            lambda x: x['Amount in Transaction Currency'] if is_real_data(x) else "", 
            axis=1
        )
        df_sheet['Fiscal Period'] = df_sheet.apply(
            lambda x: MES_NUMERO if is_real_data(x) else "", 
            axis=1
        )
    
    # Formatear fechas Día/Mes/Año
    for date_col in ['Posting Date', 'Value date', 'Clearing Date']:
        if date_col in df_sheet.columns:
            df_sheet[date_col] = df_sheet[date_col].apply(clean_date)
            
    for col in COLUMNAS_SALIDA:
        if col not in df_sheet.columns: df_sheet[col] = ""
    df_sheet = df_sheet[COLUMNAS_SALIDA]
    
    df_sheet.to_excel(writer, sheet_name=entidad, index=False)
    
    worksheet = writer.sheets[entidad]
    for i, col in enumerate(df_sheet.columns):
        max_len = max(df_sheet[col].astype(str).map(len).max(), len(col)) + 1
        
        if col in ['Amount in Transaction Currency', 'Amount in Company Code Currency']:
            worksheet.set_column(i, i, max_len, fmt_jam)
        elif col == 'Amount in Global Currency':
            worksheet.set_column(i, i, max_len, fmt_usd)
        elif 'Amount' in col:
            worksheet.set_column(i, i, max_len, fmt_std)
        else:
            worksheet.set_column(i, i, max_len)

writer.close()
print(f"Reporte Maestro Correcto Generado: {output_file}")

Reporte Maestro Correcto Generado: Master_11136580_December_2025.xlsx
